# 02 - Preprocessing & Patch Generation

English: This notebook demonstrates how AITEX strip images (256x4096) are split at the image level into reproducible train/val/test partitions, and sliced into 256x256 patches with negative sampling and mask alignment.

Hindi: Ye notebook image-level split aur 256x256 patch extraction process ko explain aur visualize karta hai.

## 1. Setup & Imports

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
from config import DEFECT_PATCH_THRESHOLD, PATCH_SIZE, SPLIT_FILE
from dataset import PatchDataset, load_split
from split_dataset import create_splits
from utils import iter_patch_origins, pad_to_patch_grid, read_image

## 2. Image-Level Dataset Split

English: Split rule: Split images first, then slice patches. Never slice patches and then randomly split, which causes severe data leakage.

Hindi: Pehle images ka train/val/test split hota hai, phir patches bante hain taaki roll/strip leakage na ho.

In [ ]:
splits = load_split()
print("Split counts (Images):")
for split_name, paths in splits.items():
    print(f"  {split_name:<8} : {len(paths)} images")

## 3. Patch Generation & Negative Sampling Caps

English: To prevent thousands of clean background patches from swamping the defective patches:
- Normal images contribute at most 4 patches (`MAX_NORMAL_PATCHES_PER_IMAGE`).
- Defective images contribute true defective patches + up to 2 hard negative background patches.

Hindi: Clean patches ki bheed se model ko bachane ke liye normal images se max 4 aur defect images se max 2 hard negatives liye jaate hain.

In [ ]:
train_binary = PatchDataset(splits["train"], stage="binary", train=True)
val_binary = PatchDataset(splits["val"], stage="binary", train=False)

print(f"Train patches (binary) : {len(train_binary)}")
print(f"Train distribution      : {train_binary.distribution()}")
print(f"Train patch sources     : {train_binary.source_distribution()}")
print(f"Val patches (binary)   : {len(val_binary)}")
print(f"Val distribution        : {val_binary.distribution()}")

## 4. Visualizing Extracted Patches

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    tensor, label = train_binary[i * 10]
    img = tensor.permute(1, 2, 0).numpy()
    # Unnormalize for visualization
    img = np.clip(img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]), 0, 1)
    ax.imshow(img)
    ax.set_title(f"Label: {'Defective' if label == 1 else 'Normal'}")
    ax.axis("off")
plt.tight_layout()
plt.show()